# Todo

1. cell2location recommends to remove mt_genes, I didn't do in the prep I did it in the Cell2location_test.ipnyb script. In future iterations I should do that here

# Packages

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import scanpy as sc
import spatialdata as spd
import mygene


/Users/janzules/miniforge3/envs/spatial_cpu_py311/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/janzules/miniforge3/envs/spatial_cpu_py311/lib/python3.11/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


# Data

In [13]:
proj_folder  = Path("/Users/janzules/Roselab/Spatial/CAR_T/")
data_folder  = proj_folder / "data"
zarr_loc     = data_folder / "zarrFiles/CART_centroid"
sc_ref_loc   = data_folder / "sc-reference/sc_reference_cell2location.h5ad"

# Output location
results_folder = proj_folder / "Results/cell2location"
ref_run_name   = results_folder / "reference_signatures"
run_name       = results_folder / "cell2location_map"


(results_folder / "gene_id_maps").mkdir(parents=True, exist_ok=True)
(results_folder / "adata").mkdir(parents=True, exist_ok=True)
(results_folder / "qc").mkdir(parents=True, exist_ok=True)

spatial_prefix = results_folder / "gene_id_maps" / "spatial_symbols_to_ensembl"

ref_prefix = results_folder / "gene_id_maps" / "ref_symbols_to_ensembl"

In [3]:
zarr_spatial = spd.read_zarr(zarr_loc)
sdata        = zarr_spatial.tables["segmentation_counts"]
del zarr_spatial

adata_ref    = sc.read_h5ad(sc_ref_loc)

print(sdata)
print(adata_ref)

/var/folders/np/h5smkry919g1mwf1wtltfqtm0000gp/T/ipykernel_74961/1142002765.py:1: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  zarr_spatial = spd.read_zarr(zarr_loc)


AnnData object with n_obs × n_vars = 427296 × 19059
    obs: 'sample', 'cell_id', 'region', 'TMA', 'mouse', 'tissue', 'condition', 'tumor_loc', 'replicate_num'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'
AnnData object with n_obs × n_vars = 135379 × 22949
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'Barcode', 'Library', 'condition', 'file_name', 'percent.mt', 'unintegrated_clusters', 'seurat_clusters', 'harmony_clusters', 'sctype_classification', 'seurat.cluster.ann', 'treatment', 'cell_type', 'UMAP_R_1', 'UMAP_R_2'
    obsm: 'X_umap_R'


In [4]:
# Double check names
def guess_gene_namespace(var_names: pd.Index) -> str:
    s = var_names.astype(str)
    if s.str.startswith("ENSMUSG").mean() > 0.95:
        return "mouse_ensembl"
    if s.str.startswith("ENSG").mean() > 0.95:
        return "human_ensembl"
    return "symbols_or_mixed"

print("spatial:", guess_gene_namespace(sdata.var_names))
print("ref:",    guess_gene_namespace(adata_ref.var_names))

spatial: symbols_or_mixed
ref: symbols_or_mixed


# Functions

In [5]:
def save_run_metadata(out_json: Path, extra: dict | None = None):
    import platform
    import anndata
    import mygene

    meta = {
        "python": platform.python_version(),
        "scanpy": sc.__version__,
        "anndata": anndata.__version__,
        "mygene": mygene.__version__,
        "date_utc": pd.Timestamp.utcnow().isoformat(),
    }
    if extra:
        meta.update(extra)
    out_json.write_text(json.dumps(meta, indent=2))

In [ ]:
def extract_ensembl_gene_ids(x):
    if x is None:
        return set()
    if isinstance(x, float) and np.isnan(x):
        return set()
    if isinstance(x, dict):
        g = x.get("gene")
        return {g} if g else set()
    if isinstance(x, list):
        out = set()
        for d in x:
            if isinstance(d, dict) and d.get("gene"):
                out.add(d["gene"])
        return out
    return set()

def resolve_unique_gene_id(df):
    genes = set()
    for x in df["ensembl"].tolist():
        genes |= extract_ensembl_gene_ids(x)
    return list(genes)[0] if len(genes) == 1 else np.nan

def map_symbols_to_mouse_ensembl(symbols: list[str], out_prefix: Path, force: bool = False) -> pd.DataFrame:
    """
    Returns: resolved dataframe with columns ['query', 'gene_id'] where gene_id is ENSMUSG... or NaN.
    Writes: sym hits, alias hits, resolved, qc json.
    """
    out_resolved = out_prefix.with_suffix(".resolved.tsv")
    out_sym = out_prefix.with_suffix(".sym.tsv")
    out_alias = out_prefix.with_suffix(".alias.tsv")
    out_qc = out_prefix.with_suffix(".qc.json")

    if out_resolved.exists() and not force:
        resolved = pd.read_csv(out_resolved, sep="\t")
        return resolved

    # Pass 1: symbol only
    res_sym = mg.querymany(
        symbols,
        scopes="symbol",
        fields="ensembl.gene,symbol",
        species="mouse",
        verbose=True,
    )
    df_sym = pd.DataFrame(res_sym)
    df_sym.to_csv(out_sym, sep="\t", index=False)

    sym_resolved = (
        df_sym.groupby("query", sort=False)
              .apply(resolve_unique_gene_id, include_groups=False)
              .rename("gene_id_sym")
              .reset_index()
    )

    # Pass 2: alias only for notfound
    # FIX: 'notfound' column is absent when all symbols are found; .get() returns scalar False
    # which causes .loc[False, ...] to mis-index rather than return an empty list.
    if "notfound" not in df_sym.columns:
        notfound = []
    else:
        notfound = df_sym.loc[df_sym["notfound"] == True, "query"].dropna().unique().tolist()

    res_alias = mg.querymany(
        notfound,
        scopes="alias",
        fields="ensembl.gene,symbol",
        species="mouse",
        verbose=True,
    )
    df_alias = pd.DataFrame(res_alias)
    df_alias.to_csv(out_alias, sep="\t", index=False)

    alias_resolved = (
        df_alias.groupby("query", sort=False)
                .apply(resolve_unique_gene_id, include_groups=False)
                .rename("gene_id_alias")
                .reset_index()
    )

    # combine with strict priority + collision guard
    resolved = sym_resolved.merge(alias_resolved, on="query", how="left")
    resolved["gene_id"] = resolved["gene_id_sym"].combine_first(resolved["gene_id_alias"])

    symbol_gene_ids = set(resolved.loc[resolved["gene_id_sym"].notna(), "gene_id_sym"])
    alias_only_mask = resolved["gene_id_sym"].isna() & resolved["gene_id"].notna()
    collides = alias_only_mask & resolved["gene_id"].isin(symbol_gene_ids)
    resolved.loc[collides, "gene_id"] = np.nan

    # only accept ENSMUSG
    is_ensmusg = resolved["gene_id"].astype("string").str.startswith("ENSMUSG", na=False)
    resolved.loc[~is_ensmusg, "gene_id"] = np.nan

    resolved = resolved[["query", "gene_id"]]
    resolved.to_csv(out_resolved, sep="\t", index=False)

    qc = {
        "n_symbols": len(symbols),
        "mapped_frac": float(resolved["gene_id"].notna().mean()),
        "n_mapped": int(resolved["gene_id"].notna().sum()),
        "n_unmapped": int(resolved["gene_id"].isna().sum()),
        "n_duplicate_gene_id_excl_na": int(
            resolved.loc[resolved["gene_id"].notna(), "gene_id"].duplicated().sum()
        ),
    }
    out_qc.write_text(json.dumps(qc, indent=2))
    return resolved

# Data Prep

## Creating Compatible name

In [7]:
# Creating the column that will match the sc reference
sdata.obs['treatment'] = sdata.obs['condition']

tumor_locations = [1, 2]

for tumor in tumor_locations:

    if tumor == 1:
        tumor_num = 1
        suffix = "_Tu1"
    elif tumor == 2:
        tumor_num = 2
        suffix = "_Tu2"
    else:
        ValueError("Something went wrong")
        
    tum_loc_mask = sdata.obs["tumor_loc"] == tumor_num
    
    sdata.obs.loc[tum_loc_mask, 'treatment'] = (
        sdata.obs.loc[tum_loc_mask, 'treatment']
        .str.replace(r"T72", "TAG72")
        + suffix
    )
# sdata.obs['treatment']


## Ensembl naming

In [8]:
import mygene
mg = mygene.MyGeneInfo()

### sdata

In [9]:

resolved_sp = map_symbols_to_mouse_ensembl(
    symbols=sdata.var_names.astype(str).tolist(),
    out_prefix=spatial_prefix,
    force=False
)

# Build sdata_c2l
symbol_to_gene = dict(zip(resolved_sp["query"], resolved_sp["gene_id"]))
mapped_ids = pd.Index(sdata.var_names).map(symbol_to_gene)

keep = ~pd.isna(mapped_ids)
sdata_c2l = sdata[:, keep].copy()

sdata_c2l.var["SYMBOL"] = sdata_c2l.var_names
sdata_c2l.var_names = pd.Index(mapped_ids[keep])

# Sanity checks
assert sdata_c2l.var_names.is_unique
xmin = sdata_c2l.X.min()
assert xmin >= 0, f"Found negative values in X: min={xmin}"

# Save gene lists + a small QC report
(pd.Series(sdata.var_names, name="SYMBOL")
   .to_csv(results_folder / "qc" / "spatial_symbols_all.tsv", sep="\t", index=False))
(pd.Series(sdata_c2l.var["SYMBOL"].values, name="SYMBOL")
   .to_csv(results_folder / "qc" / "spatial_symbols_used.tsv", sep="\t", index=False))
(pd.Series(sdata_c2l.var_names, name="ENSMUSG")
   .to_csv(results_folder / "qc" / "spatial_ensembl_used.tsv", sep="\t", index=False))

unmapped = resolved_sp.loc[resolved_sp["gene_id"].isna(), "query"].sort_values()
unmapped.to_csv(results_folder / "qc" / "spatial_unmapped_symbols.tsv", sep="\t", index=False, header=["SYMBOL"])

# Save the cell2location-ready spatial AnnData table
sdata_c2l.write_h5ad(results_folder / "adata" / "spatial_sdata_c2l.h5ad")

save_run_metadata(results_folder / "qc" / "run_metadata.json", extra={"dataset": "spatial"})
print("spatial_sdata_c2l:", sdata_c2l.shape)
print("unmapped spatial symbols:", len(unmapped))


spatial_sdata_c2l: (427296, 18947)
unmapped spatial symbols: 112


/var/folders/np/h5smkry919g1mwf1wtltfqtm0000gp/T/ipykernel_74961/1282166229.py:8: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  "scanpy": sc.__version__,
/var/folders/np/h5smkry919g1mwf1wtltfqtm0000gp/T/ipykernel_74961/1282166229.py:9: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  "anndata": anndata.__version__,


### adata_ref

In [10]:
resolved_ref = map_symbols_to_mouse_ensembl(
    symbols=adata_ref.var_names.astype(str).tolist(),
    out_prefix=ref_prefix,
    force=False
)

In [64]:
import numpy as np
import pandas as pd
from scipy import sparse
import anndata as ad

# ----------------------------
# Build reference c2l view (ENSMUSG) with SUM-collapsing duplicates
# Requires:
#   - adata_ref: AnnData, var_names are symbols
#   - ref_symbol_to_gene: dict {symbol -> ENSMUSG}
# Optional:
#   - sdata_c2l: AnnData with ENSMUSG var_names (for overlap printout)
# ----------------------------

# 1) Map symbols -> ENSMUSG (preserves order)
ref_mapped_ids = pd.Index(adata_ref.var_names.astype(str)).map(ref_symbol_to_gene)

# 2) Keep only successfully mapped genes
ref_keep = ~pd.isna(ref_mapped_ids)
adata_ref_mapped = adata_ref[:, ref_keep].copy()

# Store original symbol + mapped gene_id metadata (still aligned to X columns)
adata_ref_mapped.var["SYMBOL"]  = adata_ref_mapped.var_names.astype(str)
adata_ref_mapped.var["gene_id"] = pd.Index(ref_mapped_ids[ref_keep]).astype(str).values

# 3) SUM-collapse duplicates by gene_id
gid = adata_ref_mapped.var["gene_id"].astype(str).values
uniq, inv = np.unique(gid, return_inverse=True)

X = adata_ref_mapped.X
if not sparse.issparse(X):
    X = sparse.csr_matrix(X)

# Create mapping matrix M so X_sum = X @ M sums columns by gene_id
M = sparse.csr_matrix(
    (np.ones_like(inv, dtype=X.dtype), (np.arange(len(inv)), inv)),
    shape=(len(inv), len(uniq))
)
X_sum = X @ M  # (cells x unique_genes)

# 4) Build output AnnData
adata_ref_c2l = ad.AnnData(
    X=X_sum,
    obs=adata_ref_mapped.obs.copy(),
    var=pd.DataFrame(index=pd.Index(uniq, name="gene_id")),
)

# Representative symbol per gene_id (highest total counts among the duplicates)
colsum = np.asarray(X.sum(axis=0)).ravel()
var_tmp = pd.DataFrame({
    "gene_id": gid,
    "SYMBOL": adata_ref_mapped.var["SYMBOL"].values,
    "colsum": colsum
})

rep = (var_tmp.sort_values(["gene_id", "colsum"], ascending=[True, False])
              .drop_duplicates("gene_id")
              .set_index("gene_id"))

adata_ref_c2l.var["SYMBOL"] = rep.loc[adata_ref_c2l.var_names, "SYMBOL"].values

# (Optional) carry over other .var columns if you want; safest is to NOT,
# because duplicates make most annotations ambiguous.

# 5) Ensure correct naming
adata_ref_c2l.var_names = pd.Index(uniq)  # ENSMUSG IDs
assert adata_ref_c2l.var_names.is_unique

print("adata_ref (orig):        ", adata_ref.shape)
print("adata_ref_mapped:        ", adata_ref_mapped.shape, f"(mapped genes: {ref_keep.sum()}/{adata_ref.n_vars})")
print("adata_ref_c2l (summed):  ", adata_ref_c2l.shape, f"(unique ENSMUSG: {adata_ref_c2l.n_vars})")

# 6) Optional: overlap with spatial c2l view
if "sdata_c2l" in globals():
    shared = adata_ref_c2l.var_names.intersection(sdata_c2l.var_names)
    print("shared ENSMUSG with sdata_c2l:", len(shared))


adata_ref (orig):         (135379, 22949)
adata_ref_mapped:         (135379, 21864) (mapped genes: 21864/22949)
adata_ref_c2l (summed):   (135379, 21855) (unique ENSMUSG: 21855)
shared ENSMUSG with sdata_c2l: 16784


# Old Code

In [11]:
print("ref mapped fraction:", resolved_ref["gene_id"].notna().mean())
print("ref unmapped:", resolved_ref["gene_id"].isna().sum())

ref mapped fraction: 0.9527212514706523
ref unmapped: 1085


In [38]:
ref_symbol_to_gene = dict(zip(resolved_ref["query"], resolved_ref["gene_id"]))
# pandas.Index.map creates a stable behavior, pd.index forces things to be in index and not any other dtyep for Index.map() function
# .map() for each element in the index, look it up in the dict, if found, rturn a the value, if not found NA. 
# So, any genes that doesn't have a match, will have a false an NA where there is no match)
ref_mapped_ids = pd.Index(adata_ref.var_names).map(ref_symbol_to_gene)

print("Dimension of adata_ref:", adata_ref.shape)
print("Length of mapped symbols dict", len(ref_symbol_to_gene))

Dimension of adata_ref: (135379, 22949)
Length of mapped symbols dict 22949


In [21]:
# ref_keep has the same order of the index in the original index from the adata_ref
# ref_keep is now a True/False np.array that will select where the genes that have been mapped are and are not. 
# used for subsetting the adata_ref based on mapping results.
ref_keep = ~pd.isna(ref_mapped_ids)

In [42]:
print("length of genes for keeping:", ref_keep.sum())
print("how may genes were not in the mapping:", np.size(ref_keep) - np.sum(ref_keep))

length of genes for keeping: 21864
how may genes were not in the mapping: 1085


In [43]:
#Keeping only the genes that have an ensembl mapping 
adata_ref_c2l = adata_ref[:, ref_keep].copy()

In [46]:
# Keep original symbol, store mapped ENSMUSG as a column
adata_ref_c2l.var["SYMBOL"]  = adata_ref_c2l.var_names.astype(str)
# It assigns each retained gene column in adata_ref_c2l its mapped Ensembl gene ID, in the same order as the expression matrix columns.
# The .astype(str).values forces clean string IDs and positional assignment, avoiding index-based realignment or dtype issues.
adata_ref_c2l.var["gene_id"] = pd.Index(ref_mapped_ids[ref_keep]).astype(str).values

In [58]:
adata_ref_c2l.shape[1]

21864

In [61]:
import random
x = random.sample(range(0,adata_ref_c2l.shape[1]), k=20)
adata_ref_c2l.var.iloc[x,:]

,SYMBOL,gene_id
Gm13431,Gm13431,ENSMUSG00000087187
Slc44a5,Slc44a5,ENSMUSG00000028360
3110045C21Rik,3110045C21Rik,ENSMUSG00000097503
Gm11846,Gm11846,ENSMUSG00000085909
Gm1527,Gm1527,ENSMUSG00000074655
Gm27211,Gm27211,ENSMUSG00000098875
9530059O14Rik,9530059O14Rik,ENSMUSG00000097736
Mfap2,Mfap2,ENSMUSG00000060572
Pglyrp3,Pglyrp3,ENSMUSG00000042244
Gm568,Gm568,ENSMUSG00000048824


In [ ]:
ref_keep = ~pd.isna(ref_mapped_ids)
adata_ref_c2l = adata_ref[:, ref_keep].copy()

adata_ref_c2l.var["SYMBOL"] = adata_ref_c2l.var_names
adata_ref_c2l.var_names = pd.Index(ref_mapped_ids[ref_keep])

assert adata_ref_c2l.var_names.is_unique

AssertionError: 

In [13]:
shared = sdata_c2l.var_names.intersection(adata_ref.var_names)
print("shared genes:", len(shared))

sdata_c2l = sdata_c2l[:, shared].copy()
adata_ref_c2l = adata_ref[:, shared].copy()

# Save shared gene list + processed reference
pd.Series(shared, name="ENSMUSG").to_csv(results_folder / "qc" / "shared_genes.tsv", sep="\t", index=False)

adata_ref_c2l.write_h5ad(results_folder / "adata" / "sc_reference_c2l.h5ad")
sdata_c2l.write_h5ad(results_folder / "adata" / "spatial_sdata_c2l_shared.h5ad")

print("spatial after shared:", sdata_c2l.shape)
print("ref after shared:", adata_ref_c2l.shape)


shared genes: 0
spatial after shared: (427296, 0)
ref after shared: (135379, 0)


In [14]:
qc = {
    "spatial_mapped_genes": int(sdata_c2l.n_vars),
    "ref_genes": int(adata_ref.n_vars),
    "shared_genes": int(len(shared)),
}
(results_folder / "qc" / "c2l_input_qc.json").write_text(json.dumps(qc, indent=2))


74

In [15]:
qc

{'spatial_mapped_genes': 0, 'ref_genes': 22949, 'shared_genes': 0}

#### Testing sc ref dupes

In [20]:
ref_symbols_kept = pd.Index(adata_ref.var_names[ref_keep]).astype(str)
ref_ens_kept = pd.Index(ref_mapped_ids[ref_keep]).astype("string")

dup_mask = ref_ens_kept.duplicated(keep=False)
dups = pd.DataFrame({
    "symbol": ref_symbols_kept[dup_mask].values,
    "ensembl": ref_ens_kept[dup_mask].values,
}).sort_values(["ensembl", "symbol"])

dups.head(40), dups["ensembl"].nunique(), len(dups)


(           symbol             ensembl
 10        Gm11492  ENSMUSG00000020486
 3           Sept4  ENSMUSG00000020486
 9            PSCA  ENSMUSG00000022598
 5            Psca  ENSMUSG00000022598
 4            PISD  ENSMUSG00000023452
 0            Pisd  ENSMUSG00000023452
 7   4930430A15Rik  ENSMUSG00000027157
 12        Gm13941  ENSMUSG00000027157
 11        Gm13942  ENSMUSG00000027157
 13        Gm15130  ENSMUSG00000027157
 1   1500011B03Rik  ENSMUSG00000072694
 2   2610524H06Rik  ENSMUSG00000072694
 6          Gm9780  ENSMUSG00000095304
 14         Plac9a  ENSMUSG00000095304
 8          Plac9b  ENSMUSG00000095304,
 6,
 15)

In [21]:
ref_keep = ~pd.isna(ref_mapped_ids)
adata_ref_c2l = adata_ref[:, ref_keep].copy()

adata_ref_c2l.var["SYMBOL"] = adata_ref_c2l.var_names
adata_ref_c2l.var_names = pd.Index(ref_mapped_ids[ref_keep])

assert adata_ref_c2l.var_names.is_unique


AssertionError: 

# Choosing to sum or select the greatest of the duplicates

In [62]:
import numpy as np
import pandas as pd
from scipy import sparse

def _get_dense_col(X, j):
    """Return a dense 1D np.array for column j from dense or sparse matrix."""
    if sparse.issparse(X):
        return X[:, j].toarray().ravel()
    return np.asarray(X[:, j]).ravel()

def _pair_metrics(a, b, eps=1e-12):
    """Metrics to decide if two gene columns are redundant vs split."""
    # correlation (Pearson) on raw counts
    if a.std() < eps or b.std() < eps:
        corr = np.nan
    else:
        corr = np.corrcoef(a, b)[0, 1]

    # non-overlap: fraction of nonzero events that are exclusive to one column
    a_nz = a > 0
    b_nz = b > 0
    union = (a_nz | b_nz).sum()
    if union == 0:
        frac_exclusive = np.nan
    else:
        exclusive = (a_nz ^ b_nz).sum()
        frac_exclusive = exclusive / union

    # overlap of counts (Jaccard on nonzero pattern)
    inter = (a_nz & b_nz).sum()
    jaccard_nz = inter / union if union else np.nan

    # how much signal is in the "smaller" column
    sum_a = a.sum()
    sum_b = b.sum()
    frac_smaller_mass = (min(sum_a, sum_b) / max(sum_a, sum_b)) if max(sum_a, sum_b) > 0 else np.nan

    return corr, frac_exclusive, jaccard_nz, frac_smaller_mass, sum_a, sum_b

# find duplicated gene_ids
gid = adata_ref_c2l.var["gene_id"].astype(str).values
dup_mask = pd.Index(gid).duplicated(keep=False)
dup_gids = pd.Series(gid[dup_mask]).value_counts()

print("Duplicated ENSMUSG IDs:", (dup_gids > 1).sum())
print("Total duplicate columns involved:", dup_mask.sum())
print("Top dup multiplicities:")
display(dup_gids.head(10))

rows = []
X = adata_ref_c2l.X

for g, k in dup_gids.items():
    if k < 2:
        continue
    cols = np.where(gid == g)[0]

    # compute metrics for all pairs within this gene_id group
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            ci, cj = cols[i], cols[j]
            a = _get_dense_col(X, ci)
            b = _get_dense_col(X, cj)

            corr, frac_excl, jac, frac_small_mass, sum_a, sum_b = _pair_metrics(a, b)

            rows.append({
                "gene_id": g,
                "col_i": ci,
                "col_j": cj,
                "symbol_i": adata_ref_c2l.var_names[ci],
                "symbol_j": adata_ref_c2l.var_names[cj],
                "corr": corr,
                "frac_exclusive_nz": frac_excl,
                "jaccard_nz": jac,
                "frac_smaller_mass": frac_small_mass,
                "sum_i": sum_a,
                "sum_j": sum_b,
            })

dup_report = pd.DataFrame(rows).sort_values(
    ["frac_exclusive_nz", "corr"], ascending=[False, True]
)

display(dup_report.head(30))


Duplicated ENSMUSG IDs: 6
Total duplicate columns involved: 15
Top dup multiplicities:


ENSMUSG00000027157    4
ENSMUSG00000095304    3
ENSMUSG00000023452    2
ENSMUSG00000072694    2
ENSMUSG00000020486    2
ENSMUSG00000022598    2
Name: count, dtype: int64

,gene_id,col_i,col_j,symbol_i,symbol_j,corr,frac_exclusive_nz,jaccard_nz,frac_smaller_mass,sum_i,sum_j
12,ENSMUSG00000022598,16045,19695,Psca,PSCA,-0.000274,1.000000,0.000000,0.011494,348,4
2,ENSMUSG00000027157,18037,20546,4930430A15Rik,Gm15130,-0.000072,1.000000,0.000000,0.040816,49,2
0,ENSMUSG00000027157,18037,20544,4930430A15Rik,Gm13942,-0.000072,1.000000,0.000000,0.040816,49,2
1,ENSMUSG00000027157,18037,20545,4930430A15Rik,Gm13941,-0.000072,1.000000,0.000000,0.040816,49,2
6,ENSMUSG00000095304,17275,18453,Gm9780,Plac9b,-0.000067,1.000000,0.000000,0.437500,16,7
7,ENSMUSG00000095304,17275,21839,Gm9780,Plac9a,-0.000036,1.000000,0.000000,0.125000,16,2
8,ENSMUSG00000095304,18453,21839,Plac9b,Plac9a,-0.000028,1.000000,0.000000,0.285714,7,2
4,ENSMUSG00000027157,20544,20546,Gm13942,Gm15130,-0.000015,1.000000,0.000000,1.000000,2,2
5,ENSMUSG00000027157,20545,20546,Gm13941,Gm15130,-0.000015,1.000000,0.000000,1.000000,2,2
3,ENSMUSG00000027157,20544,20545,Gm13942,Gm13941,-0.000015,1.000000,0.000000,1.000000,2,2


In [63]:
# thresholds you can tune
CORR_HIGH = 0.95
EXCL_HIGH = 0.40        # above this suggests "split"
SMALL_MASS_TINY = 0.05  # smaller column has <5% of mass -> likely redundant/noise

gene_decisions = []

for g in dup_gids.index:
    sub = dup_report[dup_report["gene_id"] == g].copy()
    if sub.empty:
        continue

    # conservative: if ANY pair is very non-overlapping, consider sum
    any_split = (sub["frac_exclusive_nz"] > EXCL_HIGH).any()

    # if duplicates are extremely correlated and exclusive fraction low, keep-one
    all_high_corr = (sub["corr"] >= CORR_HIGH).all(skipna=True)
    low_excl = (sub["frac_exclusive_nz"] <= 0.10).all(skipna=True)

    # if one of the two columns is consistently tiny mass, keep-one
    tiny_mass = (sub["frac_smaller_mass"] < SMALL_MASS_TINY).all(skipna=True)

    if any_split and not (all_high_corr and low_excl):
        decision = "SUM"
    else:
        decision = "KEEP_ONE"

    gene_decisions.append({
        "gene_id": g,
        "n_cols": int(dup_gids[g]),
        "decision": decision,
        "max_frac_exclusive_nz": sub["frac_exclusive_nz"].max(),
        "min_corr": sub["corr"].min(),
        "min_frac_smaller_mass": sub["frac_smaller_mass"].min(),
    })

decision_df = pd.DataFrame(gene_decisions).sort_values(
    ["decision", "max_frac_exclusive_nz"], ascending=[True, False]
)

display(decision_df)
print(decision_df["decision"].value_counts())


,gene_id,n_cols,decision,max_frac_exclusive_nz,min_corr,min_frac_smaller_mass
0,ENSMUSG00000027157,4,SUM,1.000000,-0.000072,0.040816
1,ENSMUSG00000095304,3,SUM,1.000000,-0.000067,0.125000
5,ENSMUSG00000022598,2,SUM,1.000000,-0.000274,0.011494
4,ENSMUSG00000020486,2,SUM,0.998043,0.009860,0.010249
3,ENSMUSG00000072694,2,SUM,0.942609,0.057866,0.724796
2,ENSMUSG00000023452,2,SUM,0.755058,0.181832,0.324589


decision
SUM    6
Name: count, dtype: int64


# Checking overlap

In [65]:
# raw, exact symbol overlap
sym_spatial = pd.Index(sdata.var_names.astype(str))
sym_ref     = pd.Index(adata_ref.var_names.astype(str))

shared_sym_exact = sym_spatial.intersection(sym_ref)

print("Exact symbol overlap:")
print("  shared symbols:", len(shared_sym_exact))
print("  spatial genes:", sdata.n_vars)
print("  ref genes:", adata_ref.n_vars)
print("  frac spatial:", len(shared_sym_exact) / sdata.n_vars)
print("  frac ref:", len(shared_sym_exact) / adata_ref.n_vars)


Exact symbol overlap:
  shared symbols: 16342
  spatial genes: 19059
  ref genes: 22949
  frac spatial: 0.857442677999895
  frac ref: 0.7121007451305068


In [66]:
sym_spatial_uc = sym_spatial.str.upper()
sym_ref_uc     = sym_ref.str.upper()

shared_sym_uc = sym_spatial_uc.intersection(sym_ref_uc)

print("Case-insensitive symbol overlap:")
print("  shared symbols:", len(shared_sym_uc))
print("  frac spatial:", len(shared_sym_uc) / sdata.n_vars)
print("  frac ref:", len(shared_sym_uc) / adata_ref.n_vars)


Case-insensitive symbol overlap:
  shared symbols: 16343
  frac spatial: 0.8574951466498767
  frac ref: 0.712144320013944


In [67]:
# requires sdata_c2l and adata_ref_c2l already built

shared_ens = sdata_c2l.var_names.intersection(adata_ref_c2l.var_names)

# original symbols that contributed to those ENSMUSG
spatial_syms_from_shared = (
    sdata_c2l.var.loc[shared_ens, "SYMBOL"]
    if "SYMBOL" in sdata_c2l.var.columns
    else pd.Series(index=shared_ens, data=shared_ens)
)

ref_syms_from_shared = adata_ref_c2l.var.loc[shared_ens, "SYMBOL"]

print("After ENSMUSG alignment:")
print("  shared ENSMUSG:", len(shared_ens))
print("  unique spatial symbols contributing:", spatial_syms_from_shared.nunique())
print("  unique ref symbols contributing:", ref_syms_from_shared.nunique())


After ENSMUSG alignment:
  shared ENSMUSG: 16784
  unique spatial symbols contributing: 16784
  unique ref symbols contributing: 16784


In [70]:
# shared symbols (case-insensitive) as a set
shared_sym_uc_set = set(pd.Index(shared_sym_uc).astype(str))

# normalize SYMBOLs to uppercase before comparison
spatial_symbols_uc = set(pd.Index(sdata_c2l.var.loc[shared_ens, "SYMBOL"]).astype(str).str.upper())
ref_symbols_uc     = set(pd.Index(adata_ref_c2l.var.loc[shared_ens, "SYMBOL"]).astype(str).str.upper())

rescued_spatial_syms = spatial_symbols_uc - shared_sym_uc_set
rescued_ref_syms     = ref_symbols_uc - shared_sym_uc_set

print("Symbols rescued by Ensembl mapping (case-insensitive):")
print("  spatial rescued:", len(rescued_spatial_syms))
print("  ref rescued:", len(rescued_ref_syms))



Symbols rescued by Ensembl mapping (case-insensitive):
  spatial rescued: 503
  ref rescued: 503


In [69]:
print("Example rescued spatial symbols:", list(sorted(rescued_spatial_syms))[:20])
print("Example rescued ref symbols:", list(sorted(rescued_ref_syms))[:20])


Example rescued spatial symbols: ['A1cf', 'A2m', 'A3galt2', 'A4galt', 'A4gnt', 'Aaas', 'Aacs', 'Aadac', 'Aadacl2', 'Aadacl3', 'Aadacl4', 'Aadat', 'Aagab', 'Aak1', 'Aamdc', 'Aamp', 'Aanat', 'Aar2', 'Aard', 'Aars']
Example rescued ref symbols: ['0610007P14Rik', '0610009O20Rik', '0610011F06Rik', '0610037L13Rik', '1110001J03Rik', '1110004E09Rik', '1110008F13Rik', '1110008L16Rik', '1110034G24Rik', '1110037F02Rik', '1190002N15Rik', '1190003K10Rik', '1500011K16Rik', '1500015O10Rik', '1600002H07Rik', '1700006E09Rik', '1700007B14Rik', '1700007G11Rik', '1700007K09Rik', '1700011A15Rik']


# Preparing for modeling

In [72]:
shared = sdata_c2l.var_names.intersection(adata_ref_c2l.var_names)
print("shared genes for model:", len(shared))

sdata_c2l_model = sdata_c2l[:, shared].copy()
adata_ref_c2l_model = adata_ref_c2l[:, shared].copy()


shared genes for model: 16784


In [73]:
print("Spatial X dtype:", sdata_c2l_model.X.dtype)
print("Ref X dtype:", adata_ref_c2l_model.X.dtype)

# Make sure counts are non-negative
assert sdata_c2l_model.X.min() >= 0
assert adata_ref_c2l_model.X.min() >= 0

# Optional: quick library sizes
import numpy as np
sp_lib = np.asarray(sdata_c2l_model.X.sum(axis=1)).ravel()
rf_lib = np.asarray(adata_ref_c2l_model.X.sum(axis=1)).ravel()
print("Spatial libsize median:", np.median(sp_lib))
print("Ref libsize median:", np.median(rf_lib))


Spatial X dtype: float32
Ref X dtype: int64
Spatial libsize median: 1277.0
Ref libsize median: 5186.0


In [74]:
import numpy as np
from scipy import sparse

def check_integer_like(X, name, n=50000):
    if sparse.issparse(X):
        data = X.data
    else:
        data = np.asarray(X).ravel()
    if data.size == 0:
        print(f"{name}: empty")
        return
    if data.size > n:
        idx = np.random.choice(data.size, n, replace=False)
        data = data[idx]
    frac = np.mean(np.isclose(data, np.round(data)))
    print(f"{name}: integer-like fraction ~ {frac:.6f} (1.0 is ideal)")
    print(f"{name}: dtype={X.dtype}, min={data.min()}, max={data.max()}")

check_integer_like(adata_ref_c2l_model.X, "ref")
check_integer_like(sdata_c2l_model.X, "spatial")


ref: integer-like fraction ~ 1.000000 (1.0 is ideal)
ref: dtype=int64, min=1, max=370
spatial: integer-like fraction ~ 1.000000 (1.0 is ideal)
spatial: dtype=float32, min=1.0, max=403.0


## Why is the min not 0?

In [75]:
import numpy as np
from scipy import sparse

def zero_fraction(X, n=200000):
    if sparse.issparse(X):
        # zeros are implicit; look at nnz
        nnz = X.nnz
        total = X.shape[0] * X.shape[1]
        return 1 - (nnz / total)
    else:
        arr = np.asarray(X)
        if arr.size > n:
            idx = np.random.choice(arr.size, n, replace=False)
            arr = arr.ravel()[idx]
        return np.mean(arr == 0)

print("Spatial zero fraction:", zero_fraction(sdata_c2l_model.X))
print("Ref zero fraction:", zero_fraction(adata_ref_c2l_model.X))


Spatial zero fraction: 0.9257132675762676
Ref zero fraction: 0.8736059609997484


In [76]:
from scipy import sparse
import numpy as np

X = sdata_c2l_model.X
if sparse.issparse(X):
    vals = np.unique(X.data)
else:
    vals = np.unique(np.asarray(X).ravel())
print("Spatial unique values (first 20):", vals[:20])


Spatial unique values (first 20): [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17. 18.
 19. 20.]


In [77]:
print("sdata_c2l_model.X type:", type(sdata_c2l_model.X))
print("Layers:", list(sdata_c2l_model.layers.keys()) if hasattr(sdata_c2l_model, "layers") else "no layers attr")
print("Has raw?", sdata_c2l_model.raw is not None)


sdata_c2l_model.X type: <class 'scipy.sparse._csr.csr_matrix'>
Layers: []
Has raw? False


# Saving data for modeling

In [78]:
from pathlib import Path

# out_dir = Path("cell2location_inputs")
# out_dir.mkdir(parents=True, exist_ok=True)

sp_path  = results_folder / "spatial_sdata_c2l_model.h5ad"
ref_path = results_folder / "ref_adata_c2l_model.h5ad"

# Ensure stable naming
sdata_c2l_model.var_names.name = "gene_id"
adata_ref_c2l_model.var_names.name = "gene_id"

# Light sanity
assert sdata_c2l_model.var_names.is_unique
assert adata_ref_c2l_model.var_names.is_unique
assert sdata_c2l_model.var_names.equals(adata_ref_c2l_model.var_names), "Gene order mismatch!"

# Write
sdata_c2l_model.write_h5ad(sp_path, compression="gzip")
adata_ref_c2l_model.write_h5ad(ref_path, compression="gzip")

print("Saved:")
print(" ", sp_path.resolve())
print(" ", ref_path.resolve())


Saved:
  /Users/janzules/Roselab/Spatial/CAR_T/code/CellTypeAnnotation/cell2location_inputs/spatial_sdata_c2l_model.h5ad
  /Users/janzules/Roselab/Spatial/CAR_T/code/CellTypeAnnotation/cell2location_inputs/ref_adata_c2l_model.h5ad


In [79]:
import json
import numpy as np
from scipy import sparse

def libsize_median(adata):
    return float(np.median(np.asarray(adata.X.sum(axis=1)).ravel()))

manifest = {
    "spatial": {
        "n_obs": int(sdata_c2l_model.n_obs),
        "n_vars": int(sdata_c2l_model.n_vars),
        "X_type": str(type(sdata_c2l_model.X)),
        "X_dtype": str(sdata_c2l_model.X.dtype),
        "zero_fraction": float(1 - (sdata_c2l_model.X.nnz / (sdata_c2l_model.n_obs * sdata_c2l_model.n_vars)))
            if sparse.issparse(sdata_c2l_model.X) else None,
        "libsize_median": libsize_median(sdata_c2l_model),
        "first_10_genes": sdata_c2l_model.var_names[:10].tolist(),
    },
    "reference": {
        "n_obs": int(adata_ref_c2l_model.n_obs),
        "n_vars": int(adata_ref_c2l_model.n_vars),
        "X_type": str(type(adata_ref_c2l_model.X)),
        "X_dtype": str(adata_ref_c2l_model.X.dtype),
        "zero_fraction": float(1 - (adata_ref_c2l_model.X.nnz / (adata_ref_c2l_model.n_obs * adata_ref_c2l_model.n_vars)))
            if sparse.issparse(adata_ref_c2l_model.X) else None,
        "libsize_median": libsize_median(adata_ref_c2l_model),
        "first_10_genes": adata_ref_c2l_model.var_names[:10].tolist(),
    },
}

manifest_path = out_dir / "model_inputs_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved manifest:", manifest_path.resolve())


Saved manifest: /Users/janzules/Roselab/Spatial/CAR_T/code/CellTypeAnnotation/cell2location_inputs/model_inputs_manifest.json


In [ ]:
#How to load
# import scanpy as sc
# from pathlib import Path

# out_dir = Path("cell2location_inputs")
# sdata_c2l_model = sc.read_h5ad(out_dir / "spatial_sdata_c2l_model.h5ad")
# adata_ref_c2l_model = sc.read_h5ad(out_dir / "ref_adata_c2l_model.h5ad")
